# SDE-Net direct multi-horizon post-hoc con label STGAN

Questo notebook **non riaddestra SDE-Net o STGAN**. Legge l'unico `predictions.csv` del modello diretto t+1,…,t+6, collega ogni canale alla decisione STGAN sulla coppia esatta `(location, timestamp target)` e rigenera l'analisi normal/rare.

La label usata è esclusivamente `anomaly_group`, derivata da `is_anomaly` di STGAN. `event_group` non viene creato. Le righe iniziali non valutate da STGAN per il context window sono escluse con un inner join e conteggiate nell'audit.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in Path.cwd().resolve().parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.experiments import sde_pipeline as pipe
from physiq_pv.reporting.pointwise_detector_posthoc import build_pointwise_detector_evaluation
print('repo root:', ROOT)

## 1. Percorsi e configurazione

Sul server si possono sovrascrivere i percorsi con `STGAN_SEED_DIR`, `SDE_MULTIHORIZON_PREDICTIONS` e `STGAN_POSTHOC_ROOT`.

In [ ]:
FORECAST_HORIZONS = pipe.FORECAST_HORIZONS
STGAN_SEED = 20
STGAN_SEED_DIR = Path(os.environ.get(
    'STGAN_SEED_DIR', ROOT / 'outputs' / 'pvgis_stgan' / 'paper_reference' / f'seed_{STGAN_SEED}'
)).resolve()
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'

# È lo stesso run diretto del notebook SDE-Net principale. STGAN sostituisce
# le label MTGFlow soltanto nella valutazione post-hoc.
BASE_CONFIG = {**pipe.DEFAULT_CONFIG,
    'name': 'paper_faithful_gaussian_detector_mtgflow_ep60',
    'forecast_horizons': ','.join(map(str, FORECAST_HORIZONS)),
    'epochs': 60, 'batch_size': 16, 'lr': 1e-4, 'lr_g': 1e-2,
    'dropout': 0.0, 'train_normal_only': False, 'anomaly_source': 'detector',
    'ood_smoke_test': True, 'sde_sigma_warmup_epochs': 30,
    'irradiance_loss_weight': 0.1, 'detector_regional_quantile': 0.975,
}
SDE_PREDICTIONS = Path(os.environ.get(
    'SDE_MULTIHORIZON_PREDICTIONS',
    ROOT / pipe.make_out_dir(BASE_CONFIG) / 'predictions.csv',
)).resolve()
EVALUATION_DIR = Path(os.environ.get(
    'STGAN_POSTHOC_ROOT', ROOT / 'outputs' / f'sde_stgan_direct_multihorizon_seed{STGAN_SEED}'
)).resolve()
MIN_MATCH_FRACTION = 0.90
RUN_RELABEL = not (EVALUATION_DIR / 'evaluation_source.json').is_file()
RUN_ANALYSIS = True
ALLOW_OVERWRITE = False

print('horizons:', FORECAST_HORIZONS)
print('STGAN scores:', STGAN_SCORES)
print('SDE direct predictions:', SDE_PREDICTIONS)
print('evaluation output:', EVALUATION_DIR)

In [ ]:
missing = [path for path in (STGAN_SCORES, SDE_PREDICTIONS) if not path.is_file()]
if missing:
    raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))
stgan_header = set(pd.read_csv(STGAN_SCORES, nrows=0).columns)
required_stgan = {'location', 'timestamp', 'anomaly_score', 'threshold', 'is_anomaly'}
if not required_stgan <= stgan_header:
    raise ValueError(f'Colonne STGAN mancanti: {sorted(required_stgan - stgan_header)}')
sde_header = set(pd.read_csv(SDE_PREDICTIONS, nrows=0).columns)
required_sde = {'location', 'timestamp', 'issue_timestamp', 'horizon_hours', 'y_true'}
if not required_sde <= sde_header or not ({'y_pred', 'y_pred_mean'} & sde_header):
    raise ValueError(f'Schema SDE-Net diretto non compatibile: {sorted(sde_header)}')
saved_horizons = tuple(sorted(pd.read_csv(
    SDE_PREDICTIONS, usecols=['horizon_hours']
)['horizon_hours'].drop_duplicates().astype(int)))
if saved_horizons != FORECAST_HORIZONS:
    raise ValueError(f'Orizzonti salvati {saved_horizons}, attesi {FORECAST_HORIZONS}.')
print('OK: unico CSV diretto t+1,...,t+6 e schema STGAN compatibile')

## 2. Join puntuale STGAN → canali diretti SDE-Net

Le label MTGFlow già presenti vengono rimosse. Una decisione STGAN `(location, timestamp)` viene replicata sulle sole righe t+h che prevedono esattamente quel target; l'orizzonte non entra nella chiave del detector.

In [ ]:
if RUN_RELABEL:
    RELABEL_RESULT = build_pointwise_detector_evaluation(
        SDE_PREDICTIONS, STGAN_SCORES, EVALUATION_DIR,
        detector_name='stgan', min_match_fraction=MIN_MATCH_FRACTION,
        allow_overwrite=ALLOW_OVERWRITE,
    )
else:
    print("RUN_RELABEL=False: uso l'output evaluation-only esistente.")

metadata_path = EVALUATION_DIR / 'evaluation_source.json'
if not metadata_path.is_file():
    raise FileNotFoundError(metadata_path)
AUDIT = json.loads(metadata_path.read_text(encoding='utf-8'))
display(pd.DataFrame([AUDIT])[[
    'detector', 'forecast_mode', 'horizons_hours', 'source_prediction_rows',
    'matched_rows', 'excluded_unmatched_rows', 'match_fraction',
    'normal_rows', 'rare_rows',
]])

In [ ]:
joined_path = EVALUATION_DIR / 'predictions.csv'
joined = pd.read_csv(joined_path, usecols=lambda c: c in {
    'location', 'timestamp', 'horizon_hours', 'anomaly_group', 'event_group'
})
assert 'anomaly_group' in joined
assert 'event_group' not in joined
assert tuple(sorted(joined['horizon_hours'].unique())) == FORECAST_HORIZONS
assert not joined.duplicated(['location', 'timestamp', 'horizon_hours']).any()
print('OK: t+1,...,t+6 usano solo anomaly_group STGAN sul target')

## 3. Analisi post-hoc dell'unico run diretto t+1,…,t+6

In [ ]:
ANALYSIS_COMMAND = pipe.build_analysis_command(
    str(EVALUATION_DIR), BASE_CONFIG, predictions=str(joined_path),
)
if RUN_ANALYSIS:
    subprocess.run(ANALYSIS_COMMAND, check=True, cwd=ROOT)
else:
    print('RUN_ANALYSIS=False: analisi non avviata.')

In [ ]:
POSTHOC_PATHS = pipe.build_direct_multihorizon_posthoc(EVALUATION_DIR)
DIRECT_METRICS = pd.read_csv(POSTHOC_PATHS['metrics'])
display(DIRECT_METRICS)
print({name: str(path) for name, path in POSTHOC_PATHS.items()})

## Interpretazione

Le curve confrontano MAE e RMSE dei sei output diretti sui campioni che **STGAN** classifica normali o anomali per la stessa località e lo stesso timestamp target. Le label sono usate soltanto dopo il forecasting: non modificano il training SDE-Net e non sono ground truth supervisionata. Il confronto con MTGFlow misura quindi la sensibilità delle conclusioni al detector non supervisionato scelto.